In [45]:
import pandas as pd
application_df = pd.read_csv('IS453 Group Assignment - Data/IS453 Group Assignment - Application Data.csv')
bureau_df = pd.read_csv('IS453 Group Assignment - Data/IS453 Group Assignment - Bureau Data.csv')

# Week 7 Preparation
Unpack the file "IS453 Group Assignment - Data.zip"<BR>
Based on the data files and data dictionary, answer the questions below.

**DIY Q4: Explore Assignment Data**
- How many rows and column in each data file?
- Through which variable is the data in two files linked?
- Is the relationship between the records in the files one-to-one or one-to-many? In which direction?
- Are all of the records in each file linked to the other? If not, which file has unlinked records?
- Which variables would potentially conflict with fair lending principles?

**Answers**
- **Size:** Application 307,511 rows x 120 columns; Bureau 1,716,428 rows x 17 columns.
- **Link:** `SK_ID_CURR`.
- **Relationship:** one-to-many, from Application to Bureau. `SK_ID_CURR` is unique in Application, while Bureau has 1 to 116 rows per ID (mean 5.6, median 4). `SK_ID_BUREAU` is unique per Bureau row.
- **Unlinked records:** Not all records link. 263,491 IDs appear in both files. 44,020 application IDs have no bureau history (these need a "no bureau record" flag/treatment after the merge). 42,320 bureau IDs have no application row, so these are unusable for modelling.
- **Fair lending concerns:** `CODE_GENDER`, `DAYS_BIRTH` (age), `NAME_FAMILY_STATUS`, `CNT_CHILDREN`, `CNT_FAM_MEMBERS` (family status/dependants), `NAME_EDUCATION_TYPE` (possible proxy), `REGION_POPULATION_RELATIVE`, `REGION_RATING_CLIENT(_W_CITY)` and the housing-block variables (geographic proxies / redlining risk), `ORGANIZATION_TYPE` and `NAME_INCOME_TYPE` (e.g. maternity leave), `NAME_TYPE_SUITE`, and the `*_CNT_SOCIAL_CIRCLE` variables (judged on others' behaviour, not the applicant's).

In [46]:
print('Application:', application_df.shape, '| Bureau:', bureau_df.shape)

app_ids, bur_ids = set(application_df.SK_ID_CURR), set(bureau_df.SK_ID_CURR)
print('Application IDs with no bureau record:', len(app_ids - bur_ids))
print('Bureau IDs with no application record:', len(bur_ids - app_ids))
print('Overall bad rate:', application_df.TARGET.mean()*100)

Application: (307511, 120) | Bureau: (1716428, 17)
Application IDs with no bureau record: 44020
Bureau IDs with no application record: 42320
Overall bad rate: 8.072881945686495


# Week 9 Preparation
Based on the group assignment data, answer the DIY question below.

**DIY Q4: Preparing the bureau data for merging**

To combine the bureau data with the application data you will need to "flatten" the multiple rows associated with each application ID (SK_ID_CURR) into a single row. <BR>
It will also be helpful if you can reduce the number of columns in the bureau data that will need to be merged.
- Based on the data dictionary definitions and the inspection of the bureau data, explain how you would propose to do feature engineering on all of the bureau variables so that they can be condensed to one value per application ID.
- Calculate the correlation between the bureau variables and use that information to further propose how columns could potentially be combined or dropped.

Put your code and answers below, and prepare to discuss in class next week.

In [47]:
import numpy as np

# Correlation on the raw bureau numeric columns
raw_corr = bureau_df.drop(columns=['SK_ID_CURR', 'SK_ID_BUREAU']).select_dtypes('number').corr()
print(raw_corr.round(2))

b = bureau_df.copy()
biz = ['Loan for business development', 'Loan for working capital replenishment', 'Loan for the purchase of equipment']
b['ACTIVE'] = (b.CREDIT_ACTIVE == 'Active').astype(int)
b['CLOSED'] = (b.CREDIT_ACTIVE == 'Closed').astype(int)
b['SOLD_BAD'] = b.CREDIT_ACTIVE.isin(['Sold', 'Bad debt']).astype(int)
b['CARD'] = (b.CREDIT_TYPE == 'Credit card').astype(int)
b['CONSUMER'] = (b.CREDIT_TYPE == 'Consumer credit').astype(int)
b['MORTGAGE'] = (b.CREDIT_TYPE == 'Mortgage').astype(int)
b['CAR'] = (b.CREDIT_TYPE == 'Car loan').astype(int)
b['MICRO'] = (b.CREDIT_TYPE == 'Microloan').astype(int)
b['BUSINESS'] = b.CREDIT_TYPE.isin(biz).astype(int)
b['OVERDUE_NOW'] = (b.CREDIT_DAY_OVERDUE > 0).astype(int)

bureau_flat = b.groupby('SK_ID_CURR').agg(
    BUR_N=('SK_ID_BUREAU', 'count'),
    BUR_N_ACTIVE=('ACTIVE', 'sum'), BUR_N_CLOSED=('CLOSED', 'sum'), BUR_N_SOLD_BAD=('SOLD_BAD', 'sum'),
    BUR_N_CARD=('CARD', 'sum'), BUR_N_CONSUMER=('CONSUMER', 'sum'), BUR_N_MORTGAGE=('MORTGAGE', 'sum'),
    BUR_N_CAR=('CAR', 'sum'), BUR_N_MICRO=('MICRO', 'sum'), BUR_N_BUSINESS=('BUSINESS', 'sum'),
    BUR_N_OVERDUE_NOW=('OVERDUE_NOW', 'sum'), BUR_MAX_DAYS_OVERDUE=('CREDIT_DAY_OVERDUE', 'max'),
    BUR_MAX_AMT_MAX_OVERDUE=('AMT_CREDIT_MAX_OVERDUE', 'max'), BUR_SUM_AMT_OVERDUE=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUR_SUM_PROLONG=('CNT_CREDIT_PROLONG', 'sum'),
    BUR_MOST_RECENT_DAYS=('DAYS_CREDIT', 'max'), BUR_OLDEST_DAYS=('DAYS_CREDIT', 'min'),
    BUR_MAX_ENDDATE=('DAYS_CREDIT_ENDDATE', 'max'), BUR_LAST_UPDATE=('DAYS_CREDIT_UPDATE', 'max'),
    BUR_SUM_CREDIT=('AMT_CREDIT_SUM', 'sum'), BUR_SUM_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUR_SUM_LIMIT=('AMT_CREDIT_SUM_LIMIT', 'sum'), BUR_SUM_ANNUITY=('AMT_ANNUITY', 'sum'),
).reset_index()
bureau_flat['BUR_DEBT_RATIO'] = bureau_flat.BUR_SUM_DEBT / bureau_flat.BUR_SUM_CREDIT.replace(0, np.nan)
bureau_flat['BUR_PCT_ACTIVE'] = bureau_flat.BUR_N_ACTIVE / bureau_flat.BUR_N
print(bureau_flat.shape, 'unique IDs:', bureau_flat.SK_ID_CURR.is_unique)

# Correlation on the flattened features: list highly correlated pairs
c = bureau_flat.drop(columns='SK_ID_CURR').corr()
pairs = c.where(np.triu(np.ones(c.shape), 1).astype(bool)).stack()
print(pairs[pairs.abs() > 0.8].sort_values(ascending=False).round(2))

                        DAYS_CREDIT  CREDIT_DAY_OVERDUE  DAYS_CREDIT_ENDDATE  \
DAYS_CREDIT                    1.00               -0.03                 0.23   
CREDIT_DAY_OVERDUE            -0.03                1.00                -0.01   
DAYS_CREDIT_ENDDATE            0.23               -0.01                 1.00   
DAYS_ENDDATE_FACT              0.88               -0.01                 0.25   
AMT_CREDIT_MAX_OVERDUE        -0.01                0.00                 0.00   
CNT_CREDIT_PROLONG            -0.03                0.00                 0.11   
AMT_CREDIT_SUM                 0.05               -0.00                 0.06   
AMT_CREDIT_SUM_DEBT            0.14               -0.00                 0.08   
AMT_CREDIT_SUM_LIMIT           0.03               -0.00                 0.10   
AMT_CREDIT_SUM_OVERDUE        -0.00                0.09                 0.00   
DAYS_CREDIT_UPDATE             0.69               -0.02                 0.25   
AMT_ANNUITY                    0.01     

**Proposed feature engineering (one row per `SK_ID_CURR`)**

| Bureau variable | Aggregation(s) | Rationale |
|---|---|---|
| `SK_ID_BUREAU` | count -> `BUR_N` | Number of prior credit lines (credit experience / activity) |
| `CREDIT_ACTIVE` | count Active / Closed / Sold+Bad debt | Sold and Bad debt are tiny (6.5k and 21 rows), so they are merged into one count. Active share shows current exposure |
| `CREDIT_CURRENCY` | **drop** | 99.9% is "currency 1", so no information |
| `CREDIT_TYPE` | count of Card, Consumer, Mortgage, Car, Microloan, **Business** (business development + working capital + equipment) | 15 types are too many and most are rare. Business credit history is directly relevant to our small-business segment |
| `DAYS_CREDIT` | max (most recent application), min (oldest = history length) | Credit recency and length of history |
| `CREDIT_DAY_OVERDUE` | max, count of lines currently overdue | Current delinquency is the strongest risk signal |
| `AMT_CREDIT_MAX_OVERDUE` | max | Worst past delinquency (65% missing; treat missing as "no overdue recorded" and consider a flag) |
| `AMT_CREDIT_SUM_OVERDUE` | sum | Total amount currently overdue |
| `CNT_CREDIT_PROLONG` | sum | Number of extensions = possible repayment stress |
| `DAYS_CREDIT_ENDDATE` | max | Longest remaining commitment |
| `DAYS_ENDDATE_FACT` | **drop** | Only exists for closed credits, and correlated 0.88 with `DAYS_CREDIT` |
| `AMT_CREDIT_SUM` / `_DEBT` / `_LIMIT` | sum; plus ratio `DEBT/CREDIT` | Total exposure and leverage. A ratio is more comparable across applicants than raw amounts |
| `AMT_ANNUITY` | sum | Instalment burden (72% missing, so interpret with care) |
| `DAYS_CREDIT_UPDATE` | max | Freshness of bureau information |

Applicants with no bureau record (44,020) will get counts = 0 and a `NO_BUREAU` flag after the left join (to add at merge time), with other aggregates left missing for binning. This avoids confusing "no history" with "clean history".

**Correlation findings**
- On the raw data: `DAYS_CREDIT` vs `DAYS_ENDDATE_FACT` is 0.88 and `DAYS_CREDIT` vs `DAYS_CREDIT_UPDATE` is 0.69, which supports dropping `DAYS_ENDDATE_FACT`. `AMT_CREDIT_SUM` vs `AMT_CREDIT_SUM_DEBT` is 0.68, which supports using the debt ratio.
- After aggregation, run the pairs check in the code cell and drop one of any pair with |r| > 0.8. In my run, `BUR_N` vs `BUR_N_CONSUMER` (0.93) and `BUR_N` vs `BUR_N_CLOSED` (0.93) came out high. Keep `BUR_N` and use the type/status mix as shares, or drop the redundant counts.

# Week 10 Consult with Prof
**Our customer segment is: Launch small business loan**

In Week 10, each group will explain to the professor their criteria for selecting rows from the data that are representative of their customer segment and the resulting:
- number of rows
- percentage of bads in comparison to the original data set

Each group will also describe its intended approach for merging the application and bureau data sets.


In [48]:
application_df = application_df.copy()     # stops pandas showing a "DataFrame is highly fragmented" warning
application_df['AGE'] = -application_df.DAYS_BIRTH / 365
segment_df = application_df[application_df.AGE.between(18, 55) & (application_df.AMT_CREDIT < 500000)].copy()
print('Rows:', len(segment_df), f'({len(segment_df) / len(application_df):.1%} of original)')
print('Bad rate: segment', f'{segment_df.TARGET.mean():.2%}', '| original', f'{application_df.TARGET.mean():.2%}')

# Left-join the flattened bureau table (from the Week 9 cell) onto the segment
merged_df = segment_df.merge(bureau_flat, on='SK_ID_CURR', how='left')
merged_df['NO_BUREAU'] = merged_df.BUR_N.isna().astype(int)
count_cols = [c for c in bureau_flat.columns if c.startswith('BUR_N')]
merged_df[count_cols] = merged_df[count_cols].fillna(0)

print('Merged shape:', merged_df.shape, '| unique IDs:', merged_df.SK_ID_CURR.is_unique)
print('No bureau record:', merged_df.NO_BUREAU.sum(), f'({merged_df.NO_BUREAU.mean():.1%})')
print(merged_df.groupby('NO_BUREAU').TARGET.agg(['size', 'mean']))

Rows: 115022 (37.4% of original)
Bad rate: segment 9.49% | original 8.07%
Merged shape: (115022, 147) | unique IDs: True
No bureau record: 18476 (16.1%)
            size      mean
NO_BUREAU                 
0          96546  0.090610
1          18476  0.117558


**Checkpoint summary: Launch small business loan**

*Row selection criteria (both approved by the professor)*
1. **Age 18 to 55**: the realistic window for starting a business. The youngest applicant is 20.5, so the lower bound removes nobody. The 55 cap removes about 22% of rows.
2. **`AMT_CREDIT` < 500,000**: a plausible startup financing range (equipment, inventory, deposits, initial operating expenses). 500k is about the median loan, so it keeps the smaller half. It leaves 115k rows and about 11k bads, enough to model, whereas lower caps shrink the sample sharply (e.g. 150k leaves 909 bads) and higher caps add loans that look less like startup financing.

*Result*
- **Rows:** 115,022 of 307,511 (37.4% of the original data)
- **Bads:** 10,920 = **9.49%**, against 8.07% in the original data (the segment is somewhat riskier than the full book)

*Merge approach*
1. Filter the application data first.
2. Flatten the bureau data to one row per `SK_ID_CURR` (Week 9 aggregations).
3. Left-join the flattened bureau table onto the filtered applications on `SK_ID_CURR`. This keeps all 115,022 applicants. The result has one row per applicant (confirmed unique).
4. 18,476 applicants (16.1%) have no bureau record. For them the count features are set to 0, the other aggregates stay missing, and a `NO_BUREAU` flag is added. These applicants have a higher bad rate (11.8% vs 9.1% with bureau history), so the flag is informative.

*Assumptions and limitations*
- No variable records the loan purpose, so age and loan size are proxies. Not every applicant in the segment is actually starting a business.
- About 19% of the filtered loans are revolving rather than term loans.
- The amount cap biases the segment towards smaller borrowers.

In [49]:
age_ok = application_df[application_df.AGE.between(18, 55)]
base_rows = (age_ok.AMT_CREDIT < 500000).sum()

rows = []
for cap in [500000, 450000, 400000, 350000, 300000, 250000, 200000, 150000, 100000]:
    sub = age_ok[age_ok.AMT_CREDIT < cap]
    rows.append({'AMT_CREDIT <': f'{cap:,}', 'Rows': len(sub), '% of 500k sample': round(len(sub) / base_rows * 100, 1),
                 'Bads': int(sub.TARGET.sum()), 'Bad rate %': round(sub.TARGET.mean() * 100, 2)})
cap_table = pd.DataFrame(rows)
print(cap_table.to_string(index=False))

AMT_CREDIT <   Rows  % of 500k sample  Bads  Bad rate %
     500,000 115022             100.0 10920        9.49
     450,000  95803              83.3  8946        9.34
     400,000  89050              77.4  8113        9.11
     350,000  81053              70.5  7252        8.95
     300,000  68103              59.2  5839        8.57
     250,000  44331              38.5  3628        8.18
     200,000  27607              24.0  2159        7.82
     150,000  13128              11.4   909        6.92
     100,000   4185               3.6   266        6.36


# Week 10 Preparation

**Do fine classing and characteristic screening on your project data**

Use the functions in the earlier part of the lab to calculate WOE and IV on a subset of your poject data.

**DIY Q4: Calculate IV for a subset of your variables**

Based on correlation to the target variable, pick the 30 characteristics with the highest correlation and calculate their IV.

In [50]:
def woe_iv(data):
    working_data = data.copy()
    working_data['Bin_Range'] = working_data['Bin_Range'].astype('object')
    working_data['Bin_Range'] = working_data['Bin_Range'].fillna('Missing')
    variable_data = pd.DataFrame()
    variable_data['Bin_Range'] = working_data.groupby(by='Bin_Range', as_index=False).count()['Bin_Range']
    variable_data['Count'] = working_data.groupby(by='Bin_Range', as_index=False).count()['BAD']
    variable_data['Events'] = working_data.groupby(by='Bin_Range', as_index=False).sum()['BAD']
    variable_data['Non_Events'] = variable_data['Count'] - variable_data['Events']
    variable_data['%_of_Events'] = variable_data['Events']/sum(variable_data['Events'])
    variable_data['%_of_Non_Events'] = variable_data['Non_Events']/sum(variable_data['Non_Events'])
    variable_data['WOE'] = np.log(variable_data['%_of_Non_Events'] / variable_data['%_of_Events'])
    variable_data['IV'] = (variable_data['%_of_Non_Events']-variable_data['%_of_Events']) * variable_data['WOE']
    variable_data['total_IV'] = variable_data['IV'].sum()
    return(variable_data)

screen_df = merged_df.drop(columns=['SK_ID_CURR', 'DAYS_BIRTH'])
corr_matrix = screen_df.select_dtypes('number').corr()
ranked = corr_matrix['TARGET'].abs().sort_values(ascending=False)
top_30 = ranked.iloc[1:31].reset_index()

screen_df['DAYS_LAST_PHONE_CHANGE'] = screen_df['DAYS_LAST_PHONE_CHANGE'].fillna(0)

def predictive_power(iv):
    if iv < 0.02:
        return 'Not useful'
    if iv < 0.1:
        return 'Weak'
    if iv < 0.3:
        return 'Medium'
    if iv < 0.5:
        return 'Strong'
    return 'Suspicious'

bin_count = 20
rows = []
for variable in top_30['index']:
    woe_bin_data = screen_df[[variable]].copy()
    woe_bin_data['BAD'] = screen_df['TARGET']

    if woe_bin_data.groupby(variable).count().shape[0] > bin_count:
        woe_bin_data['Bin_Range'] = pd.qcut(woe_bin_data[variable], q=bin_count, duplicates='drop')
    else:
        woe_bin_data['Bin_Range'] = pd.cut(woe_bin_data[variable], bins=[-1, 0, 1, 2, 3, 100])

    iv = woe_iv(woe_bin_data).loc[0, 'total_IV']
    rows.append({'Variable': variable, 'Correlation': round(corr_matrix.loc[variable, 'TARGET'], 4),
                 'IV': round(iv, 4), 'Predictive_Power': predictive_power(iv)})

iv_table = pd.DataFrame(rows).sort_values('IV', ascending=False).reset_index().drop(columns=['index'])
display(iv_table)

,Variable,Correlation,IV,Predictive_Power
0,EXT_SOURCE_1,-0.1530,0.1720,Medium
1,BUR_DEBT_RATIO,0.0961,0.0992,Weak
2,BUR_PCT_ACTIVE,0.0822,0.0895,Weak
3,BUR_OLDEST_DAYS,0.0838,0.0855,Weak
4,BUR_MOST_RECENT_DAYS,0.0450,0.0836,Weak
5,DAYS_LAST_PHONE_CHANGE,0.0667,0.0611,Weak
6,REGION_RATING_CLIENT_W_CITY,0.0645,0.0485,Weak
7,BUR_N_CLOSED,-0.0482,0.0471,Weak
8,AGE,-0.0596,0.0462,Weak
9,REGION_RATING_CLIENT,0.0612,0.0441,Weak


# Credit scorecard model (Team 6)

The cells below extend the code from previously. The model uses a stratified train/test split, learns fine-classing cut points from the training data only, calculates WOE and IV, fits logistic regression on WOE values, and converts the model into a points scorecard. `DAYS_BIRTH` and the explicitly identified fairness-sensitive fields are excluded.

## Choice of Features

### A. Preliminary IV Screening

The preliminary analysis calculated correlations and Information Value (IV) for the available variables. The variables with relatively stronger predictive relationships included:

- `EXT_SOURCE_1`
- `BUR_DEBT_RATIO`
- `BUR_PCT_ACTIVE`
- `BUR_OLDEST_DAYS`
- `DAYS_LAST_PHONE_CHANGE`
- `AMT_CREDIT`
- `DAYS_EMPLOYED`

These variables were selected as initial candidates because they had already been investigated during the team’s preliminary analysis.

### B. Business Relevance

The selected variables are also relevant to assessing applicants for a small-business loan:

| Variable | Definition | Business relevance |
|---|---|---|
| `EXT_SOURCE_1` | External credit-risk score from another data source | Indicates the applicant’s overall estimated credit risk |
| `AMT_CREDIT` | Amount of credit requested in the current application | Shows the size of the requested small-business loan |
| `DAYS_EMPLOYED` | Number of days the applicant has been employed | Indicates employment and potential income stability |
| `DAYS_ID_PUBLISH` | Number of days since the applicant’s identification document was issued | Provides an indication of applicant-profile stability |
| `DAYS_LAST_PHONE_CHANGE` | Number of days since the applicant last changed their phone number | Frequent or recent changes may indicate profile instability |
| `BUR_DEBT_RATIO` | Total bureau debt divided by total historical bureau credit | Measures the applicant’s existing debt burden |
| `BUR_PCT_ACTIVE` | Number of active bureau accounts divided by total bureau accounts | Measures the proportion of previous credit that remains active |
| `BUR_OLDEST_DAYS` | Age of the applicant’s oldest recorded bureau credit account, measured in days | Indicates the length of the applicant’s credit history |
| `BUR_N_OVERDUE_NOW` | Number of previous bureau credit accounts currently overdue | Measures current repayment problems |
| `NO_BUREAU` | Indicator showing whether the applicant has no recorded bureau history; `1` means no history and `0` means history exists | Identifies applicants whose credit risk cannot be assessed from bureau records |

### C. Fair-Lending Considerations

The following variables were excluded because they may represent protected characteristics or potential proxy variables:

- Gender
- Age-related variables
- Family status
- Number of children
- Education
- Geographic or regional variables
- Social-circle variables

Excluding these variables helps reduce the risk of unfair or discriminatory lending decisions. However, the remaining variables should still be reviewed for indirect proxy effects before the scorecard is implemented.

## 1. Set up the modelling data and select features


In [51]:
# Import the course modelling tools used for splitting, regression, and evaluation.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Select risk and credit-history variables available after the team's merge.
# Demographic and explicitly identified proxy variables are excluded.
candidate_features = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AMT_CREDIT', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH',
    'DAYS_LAST_PHONE_CHANGE', 'BUR_DEBT_RATIO', 'BUR_PCT_ACTIVE',
    'BUR_OLDEST_DAYS', 'BUR_N_OVERDUE_NOW', 'NO_BUREAU'
]
model_features = [c for c in candidate_features if c in merged_df.columns]
if len(model_features) < 5:
    raise ValueError(f'Expected at least 5 model variables, found: {model_features}')

model_df = merged_df[model_features + ['TARGET']].copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan)
# Home Credit uses 365243 as a placeholder for unknown employment duration.
model_df['DAYS_EMPLOYED'] = model_df['DAYS_EMPLOYED'].replace(365243, np.nan)
model_df['TARGET'] = model_df['TARGET'].astype(int)

# Split before learning bins to prevent test-set information leaking into training.
train_df, test_df = train_test_split(
    model_df, test_size=0.30, random_state=453, stratify=model_df['TARGET']
)
print('Model variables:', model_features)
print('Train:', train_df.shape, '| Test:', test_df.shape)
print('Train bad rate:', f"{train_df.TARGET.mean():.2%}", '| Test bad rate:', f"{test_df.TARGET.mean():.2%}")


Model variables: ['EXT_SOURCE_1', 'AMT_CREDIT', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE', 'BUR_DEBT_RATIO', 'BUR_PCT_ACTIVE', 'BUR_OLDEST_DAYS', 'BUR_N_OVERDUE_NOW', 'NO_BUREAU']
Train: (80515, 11) | Test: (34507, 11)
Train bad rate: 9.49% | Test bad rate: 9.49%


## 2. Create functions to compute training-only bins and WOE


In [52]:
# Fine classing and WOE functions based on the Week 10 techniques.
def make_train_bins(series, max_bins=10):
    # Create up to 10 equal-width intervals using training data only.
    non_missing = series.dropna()

    # A constant variable cannot be divided into meaningful ranges.
    if non_missing.nunique() <= 1:
        return np.array([])

    # Divide the observed minimum-to-maximum range evenly.
    minimum = non_missing.min()
    maximum = non_missing.max()
    edges = np.linspace(minimum, maximum, max_bins + 1)

    # Return the internal boundaries; these create max_bins intervals.
    return edges[1:-1]

def assign_bins(series, cut_points):
    # Apply the training cut points and keep missing values as a separate bin.
    if len(cut_points) == 0:
        labels = pd.Series('All', index=series.index, dtype='object')
    else:
        labels = pd.cut(
            series, bins=[-np.inf, *cut_points, np.inf], include_lowest=True
        ).astype('object')
    return labels.where(series.notna(), 'Missing').astype(str)


def fit_woe_table(binned, target):
    # Count goods and bads in each bin.
    table = pd.DataFrame({'bin': binned, 'bad': target}).groupby(
        'bin', sort=False
    )['bad'].agg(count='size', bad='sum')
    table['good'] = table['count'] - table['bad']

    # Additive smoothing prevents infinite WOE for empty good/bad bins.
    total_bad = table['bad'].sum()
    total_good = table['good'].sum()
    table['dist_bad'] = (table['bad'] + 0.5) / (total_bad + 0.5 * len(table))
    table['dist_good'] = (table['good'] + 0.5) / (total_good + 0.5 * len(table))
    table['woe'] = np.log(table['dist_good'] / table['dist_bad'])
    table['iv'] = (table['dist_good'] - table['dist_bad']) * table['woe']
    return table

# Transform every feature using WOE values learned from the training set.
bin_points = {}
woe_tables = {}
train_woe = pd.DataFrame(index=train_df.index)
test_woe = pd.DataFrame(index=test_df.index)
iv_rows = []

for feature in model_features:
    cut_points = make_train_bins(train_df[feature])
    train_bins = assign_bins(train_df[feature], cut_points)
    test_bins = assign_bins(test_df[feature], cut_points)
    table = fit_woe_table(train_bins, train_df['TARGET'])
    mapping = table['woe'].to_dict()
    train_woe[feature] = train_bins.map(mapping).fillna(0.0)
    test_woe[feature] = test_bins.map(mapping).fillna(0.0)
    bin_points[feature] = cut_points
    woe_tables[feature] = table
    iv_rows.append({'Variable': feature, 'IV': table['iv'].sum(), 'Bins': len(table)})

iv_model_table = pd.DataFrame(iv_rows).sort_values('IV', ascending=False).reset_index(drop=True)
display(iv_model_table.style.format({'IV': '{:.4f}'}))

# Fit the logistic regression model in the next cell.
lr = LogisticRegression(solver='liblinear', max_iter=1000, random_state=453)


,Variable,IV,Bins
0,EXT_SOURCE_1,0.1732,11
1,BUR_PCT_ACTIVE,0.0878,11
2,DAYS_EMPLOYED,0.0783,11
3,BUR_OLDEST_DAYS,0.0770,11
4,BUR_DEBT_RATIO,0.0692,11
5,DAYS_LAST_PHONE_CHANGE,0.0597,10
6,AMT_CREDIT,0.0343,10
7,DAYS_ID_PUBLISH,0.0211,10
8,BUR_N_OVERDUE_NOW,0.0138,7
9,NO_BUREAU,0.0119,2


| Variable | What happened | Total bins |
|---|---|---:|
| `EXT_SOURCE_1` | 10 numeric intervals + missing values | 11 |
| `BUR_PCT_ACTIVE` | 10 numeric intervals + missing values | 11 |
| `DAYS_EMPLOYED` | 10 numeric intervals + missing values | 11 |
| `BUR_OLDEST_DAYS` | 10 numeric intervals + missing values | 11 |
| `BUR_DEBT_RATIO` | 10 numeric intervals + missing values | 11 |
| `DAYS_LAST_PHONE_CHANGE` | 10 numeric intervals, no missing group | 10 |
| `AMT_CREDIT` | 10 numeric intervals, no missing group | 10 |
| `DAYS_ID_PUBLISH` | 10 numeric intervals, no missing group | 10 |
| `BUR_N_OVERDUE_NOW` | 6 non-empty numeric intervals + missing values | 7 |
| `NO_BUREAU` | 2 categories: 0 and 1 | 2 |

## 3. Fit logistic regression and calculate ROC and default probabilities for test and training data


In [53]:
# Fit logistic regression using the WOE-transformed training variables.
lr.fit(train_woe, train_df['TARGET'])

# Generate each applicant's estimated probability of default.
# These probabilities are later converted into credit scores.
train_probability = lr.predict_proba(train_woe)[:, 1]
test_probability = lr.predict_proba(test_woe)[:, 1]

# AUC evaluates how well the model ranks risky applicants above safer applicants.
# No probability cutoff is applied here because the final lending rule uses a score cutoff.
train_auc = roc_auc_score(train_df['TARGET'], train_probability)
test_auc = roc_auc_score(test_df['TARGET'], test_probability)

print(f'Training AUC: {train_auc:.4f}')
print(f'Test AUC: {test_auc:.4f}')
print('Average training default probability:', f'{train_probability.mean():.2%}')
print('Average test default probability:', f'{test_probability.mean():.2%}')


Training AUC: 0.6670
Test AUC: 0.6659
Average training default probability: 9.50%
Average test default probability: 9.52%


## 4. Function to Convert model probabilities into credit scores


In [54]:


# Course score scale: 600 score, 19:1 good-to-bad odds, 20 points per doubling.
base_score, base_good_bad_odds, points_to_double_odds = 600, 19, 20

# Higher scores represent lower predicted default risk.
factor = points_to_double_odds / np.log(2)
base_points = base_score + factor * (-lr.intercept_[0] - np.log(base_good_bad_odds))


def score_woe_frame(woe_frame):
    # Convert the model log-odds into the course credit-score scale.
    logit_bad = (
        lr.predict_log_proba(woe_frame)[:, 1]
        - lr.predict_log_proba(woe_frame)[:, 0]
    )
    return base_score + factor * (-logit_bad - np.log(base_good_bad_odds))

train_score = score_woe_frame(train_woe)
test_score = score_woe_frame(test_woe)
print('Score summary:')
print(pd.concat([
    pd.Series(train_score, name='train'),
    pd.Series(test_score, name='test')
], axis=1).describe().round(1))


Score summary:
         train     test
count  80515.0  34507.0
mean     584.6    584.5
std       18.7     18.7
min      415.3    487.8
25%      571.7    571.5
50%      583.2    583.1
75%      596.0    596.0
max      676.3    668.0


## 5. Build the points-based scorecard table


In [55]:
# Each row shows the points contributed by one variable bin.
scorecard_rows = []
for feature in model_features:
    coefficient = lr.coef_[0][model_features.index(feature)]
    table = woe_tables[feature]
    for bin_name, row in table.iterrows():
        scorecard_rows.append({
            'Variable': feature,
            'Bin': bin_name,
            'Count': int(row['count']),
            'Bad_Rate': row['bad'] / row['count'],
            'WOE': row['woe'],
            'IV': row['iv'],
            'Points': -factor * coefficient * row['woe']
        })

scorecard_table = pd.DataFrame(scorecard_rows)
print('Base points:', round(base_points, 1))
display(scorecard_table.sort_values(['Variable', 'Bin']).style.format({
    'Bad_Rate': '{:.2%}', 'WOE': '{:.4f}', 'IV': '{:.4f}', 'Points': '{:.1f}'
}))


Base points: 580.1


,Variable,Bin,Count,Bad_Rate,WOE,IV,Points
19,AMT_CREDIT,"(-inf, 90450.0]",2320,5.73%,0.5422,0.0068,17.0
11,AMT_CREDIT,"(135900.0, 181350.0]",9847,8.27%,0.1519,0.0027,4.8
13,AMT_CREDIT,"(181350.0, 226800.0]",9016,9.06%,0.0514,0.0003,1.6
18,AMT_CREDIT,"(226800.0, 272250.0]",14654,8.68%,0.0988,0.0017,3.1
15,AMT_CREDIT,"(272250.0, 317700.0]",10394,11.04%,-0.1674,0.0039,-5.2
20,AMT_CREDIT,"(317700.0, 363150.0]",6909,10.58%,-0.1205,0.0013,-3.8
14,AMT_CREDIT,"(363150.0, 408600.0]",4986,10.59%,-0.1217,0.0010,-3.8
16,AMT_CREDIT,"(408600.0, 454050.0]",9508,12.16%,-0.2770,0.0101,-8.7
12,AMT_CREDIT,"(454050.0, inf]",7295,8.91%,0.0698,0.0004,2.2
17,AMT_CREDIT,"(90450.0, 135900.0]",5586,7.09%,0.3177,0.0062,9.9


## 6. Test an approval cutoff of 560
- Evaluating to see how cutoff would have impacted historical applicants in our datatset


In [56]:
# Applicants at or above the cutoff are approved; lower scores are classified as bad.
score_cutoff = 560
for title, actual, scores in [
    ('Training', train_df['TARGET'], train_score),
    ('Test', test_df['TARGET'], test_score)
]:
    approved = scores >= score_cutoff
    predicted_bad = (~approved).astype(int)
    print(
        f'{title}: approved {approved.sum():,} ({approved.mean():.1%}); '
        f'actual defaults among approved {actual[approved].sum():,}; '
        f'actual bad rate among approved {actual[approved].mean():.2%}'
    )
    print(classification_report(
        actual, predicted_bad, target_names=['Good payer', 'Defaulter'],
        digits=3, zero_division=0
    ))


Training: approved 74,354 (92.3%); actual defaults among approved 6,339; actual bad rate among approved 8.53%
              precision    recall  f1-score   support

  Good payer      0.915     0.933     0.924     72871
   Defaulter      0.212     0.171     0.189      7644

    accuracy                          0.861     80515
   macro avg      0.563     0.552     0.557     80515
weighted avg      0.848     0.861     0.854     80515

Test: approved 31,838 (92.3%); actual defaults among approved 2,714; actual bad rate among approved 8.52%
              precision    recall  f1-score   support

  Good payer      0.915     0.933     0.924     31231
   Defaulter      0.211     0.172     0.189      3276

    accuracy                          0.860     34507
   macro avg      0.563     0.552     0.556     34507
weighted avg      0.848     0.860     0.854     34507

